# 05 · Pick-and-roll analysis

Coverage, outcome, and the tracking-derived roll/pop label.

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config as C
pd.set_option("display.width", 160, "display.max_columns", 60)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True,
                     "grid.alpha": .25, "figure.dpi": 110})
E = C.PROCESSED_DIR / "events"; D = C.DERIVED_DIR
rd  = lambda n: pd.read_parquet(D / f"{n}.parquet")
rev = lambda n: pd.read_parquet(E / f"{n}.parquet")
print("project:", C.PROJECT_ROOT.name)

project: basketball-intelligence


In [2]:
pk = rd("picks_features")
print(f"{len(pk)} picks | direct {pk.direct.mean():.1%} | {pk.pick_points.mean():.3f} pts/pick")
print("\nscreener action (derived here from tracking):")
print(pk.screener_action.value_counts(normalize=True).round(3))
pk.groupby("bhr_def_type").agg(n=("id","size"), ppp=("pick_points","mean"),
  handler=("bhr_points","mean"), screener=("scr_points","mean"), shot=("bhr_shot","mean"),
  assist=("bhr_assist","mean"), to=("bhr_turnover","mean"), foul=("bhr_foul","mean"),
  direct=("direct","mean")).sort_values("n", ascending=False).round(4)

1520 picks | direct 54.1% | 0.445 pts/pick

screener action (derived here from tracking):
screener_action
roll    0.661
stay    0.311
pop     0.028
Name: proportion, dtype: float64


,n,ppp,handler,screener,shot,assist,to,foul,direct
bhr_def_type,,,,,,,,,
over,1004,0.4681,0.3974,0.1584,0.2221,0.0857,0.0279,0.0558,0.5647
under,280,0.3714,0.3607,0.0464,0.2000,0.0571,0.0250,0.0607,0.4464
switch,218,0.4220,0.3716,0.1697,0.2202,0.0917,0.0459,0.0826,0.5459
ice,14,0.7143,0.5000,0.7143,0.1429,0.2143,0.0000,0.0000,0.7143
blitz,4,0.0000,0.0000,0.0000,0.2500,0.0000,0.0000,0.0000,0.5000


In [3]:
from scipy import stats
rng = np.random.default_rng(0)
a = pk.loc[pk.bhr_def_type=="over","pick_points"].to_numpy(float)
b = pk.loc[pk.bhr_def_type=="under","pick_points"].to_numpy(float)
dd = np.array([rng.choice(a,len(a),True).mean()-rng.choice(b,len(b),True).mean() for _ in range(4000)])
print(f"over − under = {dd.mean():+.4f} pts/pick, 95% CI [{np.percentile(dd,2.5):+.3f}, "
      f"{np.percentile(dd,97.5):+.3f}]  -> direction consistent, not separable")

over − under = +0.0971 pts/pick, 95% CI [-0.021, +0.214]  -> direction consistent, not separable


## Coverage-pair matrix

In [4]:
piv = pk.pivot_table(index="bhr_def_type", columns="scr_def_type", values="pick_points", aggfunc="mean")
cnt = pk.pivot_table(index="bhr_def_type", columns="scr_def_type", values="id", aggfunc="size")
print("counts:"); print(cnt.fillna(0).astype(int))
print("\npoints per pick (cells with n>=8):"); (piv.where(cnt>=8)).round(3)

counts:
scr_def_type  blitz  ice  none  show  soft  switch
bhr_def_type                                      
blitz             4    0     0     0     0       0
ice               0   14     0     0     0       0
over              0    0     6   502   496       0
switch            0    0     0     0     0     218
under             0    0     0   113   167       0

points per pick (cells with n>=8):


scr_def_type,blitz,ice,none,show,soft,switch
bhr_def_type,,,,,,
blitz,NaN,NaN,NaN,NaN,NaN,NaN
ice,NaN,0.714,NaN,NaN,NaN,NaN
over,NaN,NaN,NaN,0.456,0.480,NaN
switch,NaN,NaN,NaN,NaN,NaN,0.422
under,NaN,NaN,NaN,0.398,0.353,NaN


## Screen location

In [5]:
pk.groupby("location_type").agg(n=("id","size"), ppp=("pick_points","mean"),
  handler=("bhr_points","mean"), screener=("scr_points","mean"),
  switch_forced=("scr_def_type", lambda s: s.eq("switch").mean()),
  direct=("direct","mean")).round(3)

,n,ppp,handler,screener,switch_forced,direct
location_type,,,,,,
middle,979,0.473,0.410,0.154,0.141,0.566
stepUp,182,0.429,0.324,0.176,0.137,0.511
wing,359,0.376,0.357,0.100,0.153,0.490


In [6]:
rp = pk.dropna(subset=["screener_action"])
rp.groupby(["screener_action","scr_def_type"]).agg(n=("id","size"), ppp=("pick_points","mean"),
  scr_ppp=("scr_points","mean")).query("n >= 5").round(3)

n    ppp  scr_ppp
screener_action scr_def_type                     
pop             show           16  0.125    0.000
                soft           18  0.389    0.111
                switch          9  0.556    0.000
roll            ice             9  0.444    0.444
                show          420  0.440    0.198
                soft          427  0.431    0.110
                switch        142  0.380    0.190
stay            ice             5  1.200    1.200
                show          179  0.486    0.101
                soft          218  0.486    0.101
                switch         67  0.493    0.149